# Social Media Usage Survey — Hypothesis Testing & Statistical Inference

This notebook performs correlation, probability, Bayes, PMF/CDF, CLT simulation, confidence intervals, hypothesis tests, ANOVA, and linear regression using the survey data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

np.random.seed(42)


In [ ]:
FILE = "data/cleaned_social_media_survey.csv"
if not os.path.exists(FILE):
    raise FileNotFoundError("Run 01_data_inspection_preprocessing.ipynb first.")

df = pd.read_csv(FILE)
for c in ["age", "sm_hours", "active_platforms", "study_hours", "sleep_hours", "phone_last_hour", "difficulty_away"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.head()


## 1. Correlation and covariance

In [ ]:
numeric = ["sm_hours", "active_platforms", "study_hours", "sleep_hours", "phone_last_hour", "difficulty_away"]
print("Correlation matrix:")
print(df[numeric].corr())

print("\nCovariance matrix:")
print(df[numeric].cov())

# Pearson correlation: social-media hours vs study hours
r, p = stats.pearsonr(df["sm_hours"].dropna(), df.loc[df["sm_hours"].notna(), "study_hours"])
print(f"\nPearson r (social media hours vs study hours) = {r:.4f}, p = {p:.4f}")


In [ ]:
plt.scatter(df["sm_hours"], df["study_hours"])
plt.xlabel("Social media hours/day")
plt.ylabel("Study/academic hours/day")
plt.title("Social Media Usage vs Study Time")
plt.tight_layout()
plt.show()

plt.scatter(df["sm_hours"], df["sleep_hours"])
plt.xlabel("Social media hours/day")
plt.ylabel("Sleep hours/night")
plt.title("Social Media Usage vs Sleep")
plt.tight_layout()
plt.show()


## 2. Probability and Bayes theorem

Define H = more than 4 hours/day on social media and L = less than 6 hours of sleep/night.

In [ ]:
valid = df.dropna(subset=["sm_hours", "sleep_hours"]).copy()
H = valid["sm_hours"] > 4
L = valid["sleep_hours"] < 6

P_H = H.mean()
P_L = L.mean()
P_L_given_H = L[H].mean() if H.any() else np.nan
P_H_given_L = H[L].mean() if L.any() else np.nan

print(f"P(H) = {P_H:.4f}")
print(f"P(L) = {P_L:.4f}")
print(f"P(L | H) = {P_L_given_H:.4f}")
print(f"P(H | L) = {P_H_given_L:.4f}")
print(f"Bayes check = P(L|H)P(H)/P(L) = {(P_L_given_H * P_H / P_L if P_L else np.nan):.4f}")


## 3. Empirical PMF and CDF of active-platform count

In [ ]:
pmf = df["active_platforms"].value_counts(normalize=True).sort_index()
cdf = pmf.cumsum()
print("PMF:")
print(pmf)
print("\nCDF:")
print(cdf)

plt.step(cdf.index, cdf.values, where="post")
plt.xlabel("Number of active platforms")
plt.ylabel("Cumulative probability")
plt.title("Empirical CDF of Active Social Media Platforms")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()


## 4. Central Limit Theorem simulation

Use samples of n=30 and 10,000 repetitions to examine the sampling distribution of the mean social-media usage.

In [ ]:
population = df["sm_hours"].dropna().to_numpy()
n = 30
repetitions = 10000

sample_means = np.array([np.mean(np.random.choice(population, size=n, replace=True)) for _ in range(repetitions)])

print("Population mean:", population.mean())
print("Mean of simulated sample means:", sample_means.mean())
print("Standard deviation of sample means:", sample_means.std(ddof=1))

plt.hist(sample_means, bins=40, edgecolor="black")
plt.xlabel("Sample mean social-media hours")
plt.ylabel("Frequency")
plt.title("CLT Simulation (n=30, 10,000 repetitions)")
plt.tight_layout()
plt.show()


## 5. 95% confidence interval for mean social-media usage

In [ ]:
x = df["sm_hours"].dropna()
mean_x = x.mean()
sem = stats.sem(x)
ci_low, ci_high = stats.t.interval(0.95, df=len(x)-1, loc=mean_x, scale=sem)
print(f"Mean = {mean_x:.4f}")
print(f"95% CI = ({ci_low:.4f}, {ci_high:.4f})")


## 6. One-sample t-test

H0: mean daily social-media usage = 4 hours.

H1: mean daily social-media usage ≠ 4 hours.

In [ ]:
t_stat, p_value = stats.ttest_1samp(x, popmean=4)
print(f"t = {t_stat:.4f}")
print(f"p = {p_value:.4f}")


## 7. Welch two-sample t-test by gender

Compare mean social-media hours for Female and Male respondents. This test does not assume equal variances.

In [ ]:
female = df.loc[df["gender"] == "Female", "sm_hours"].dropna()
male = df.loc[df["gender"] == "Male", "sm_hours"].dropna()

t_stat, p_value = stats.ttest_ind(female, male, equal_var=False)
print(f"Female mean = {female.mean():.4f} (n={len(female)})")
print(f"Male mean = {male.mean():.4f} (n={len(male)})")
print(f"Welch t = {t_stat:.4f}, p = {p_value:.4f}")


## 8. One-way ANOVA by academic year

In [ ]:
groups = [g["sm_hours"].dropna().to_numpy() for _, g in df.groupby("academic_year") if g["sm_hours"].notna().sum() > 0]
labels = [name for name, g in df.groupby("academic_year") if g["sm_hours"].notna().sum() > 0]
F, p = stats.f_oneway(*groups)
print("Groups:", labels)
print(f"ANOVA F = {F:.4f}, p = {p:.4f}")


## 9. Linear regression: study hours predicted by social-media hours

In [ ]:
reg_df = df[["sm_hours", "study_hours"]].dropna()
X = reg_df[["sm_hours"]]
y = reg_df["study_hours"]
model = LinearRegression().fit(X, y)
pred = model.predict(X)

print(f"Intercept = {model.intercept_:.4f}")
print(f"Slope = {model.coef_[0]:.4f}")
print(f"R^2 = {model.score(X, y):.4f}")
print(f"RMSE = {mean_squared_error(y, pred, squared=False):.4f}")

plt.scatter(X["sm_hours"], y)
plt.plot(X["sm_hours"], pred)
plt.xlabel("Social media hours/day")
plt.ylabel("Study/academic hours/day")
plt.title("Linear Regression")
plt.tight_layout()
plt.show()


## 10. Behavioral frequency analysis

In [ ]:
behavior_cols = ["social_before_sleep", "study_distraction", "tried_reduce", "difficulty_away"]
for col in behavior_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))
